In [ ]:
!pip -q install xgboost scikit-learn pandas openpyxl scipy joblib

import os
import re
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier

# ============================================================
# 1) General Settings
# ============================================================
TRAIN_PATH = "train.xlsx"
VALID_PATH = "valid.xlsx"
TEST_PATH  = "test.xlsx"

TEXT_COL = "clean_text"
LABEL_DIALECT_COL = "dialect"
LABEL_HATE_COL    = "hate"
LABEL_TOPIC_COL   = "topic"
LABELS = [LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL]

RANDOM_STATE = 42
OUTPUT_DIR = "./xgboost_multilabel_with_emoji_feature"

WORD_MAX_FEATURES = 30000
CHAR_MAX_FEATURES = 20000
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 5)

XGB_N_ESTIMATORS = 300
XGB_MAX_DEPTH = 6
XGB_LR = 0.05
XGB_SUBSAMPLE = 0.8
XGB_COLSAMPLE = 0.8
XGB_REG_LAMBDA = 1.0

id2dialect = {1: "Saudi", 0: "Egyptian"}
id2hate    = {1: "Hate", 0: "Not Hate"}
id2topic   = {1: "Religious", 0: "Political"}

# ============================================================
# 2) Emoji Feature
# ============================================================
HATE_EMOJIS = set([
    '💦', '🐖', '🐷', '🐽', '👞', '🐕', '🐶', '💩', '🐄', '🐮',
    '🐑', '🐏', '👎', '😡', '🤬', '👺', '👿', '😠'
])

def emoji_flag(text: str) -> int:
    text = "" if pd.isna(text) else str(text)
    return 1 if any(e in text for e in HATE_EMOJIS) else 0

# ============================================================
# 3) Load + Validate Data - Clean Text Already Preprocessed
# ============================================================
def simple_clean_text(text: str) -> str:
    return "" if pd.isna(text) else str(text).strip()

def load_excel(path: str) -> pd.DataFrame:
    return pd.read_excel(path, engine="openpyxl")

def load_and_validate_xlsx(path: str) -> pd.DataFrame:
    df = load_excel(path)

    required = {TEXT_COL, LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {missing}\nAvailable columns: {list(df.columns)}")

    df = df.copy()
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

    for col in LABELS:
        if df[col].isna().any():
            bad_rows = df[df[col].isna()].head(5)
            raise ValueError(f"{path}: column '{col}' has NaN values. Examples:\n{bad_rows[[TEXT_COL, col]]}")

        df[col] = pd.to_numeric(df[col], errors="raise").astype(int)
        bad = ~df[col].isin([0, 1])
        if bad.any():
            examples = df.loc[bad, [TEXT_COL, col]].head(5)
            raise ValueError(f"{path}: column '{col}' must be 0/1 only. Examples:\n{examples}")

    return df

print("📂 جاري تجهيز البيانات...")
train_df = load_and_validate_xlsx(TRAIN_PATH)
valid_df = load_and_validate_xlsx(VALID_PATH)
test_df  = load_and_validate_xlsx(TEST_PATH)

print("Rows:", len(train_df), len(valid_df), len(test_df))
print("Train label counts:")
print("dialect:", train_df[LABEL_DIALECT_COL].value_counts().to_dict())
print("hate:",    train_df[LABEL_HATE_COL].value_counts().to_dict())
print("topic:",   train_df[LABEL_TOPIC_COL].value_counts().to_dict())

# ============================================================
# 4) Prepare Labels
# ============================================================
y_train = train_df[LABELS].values.astype(int)
y_valid = valid_df[LABELS].values.astype(int)
y_test  = test_df[LABELS].values.astype(int)

# ============================================================
# 5) Text + Extra Features
# ============================================================
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    max_features=WORD_MAX_FEATURES,
    min_df=2,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_NGRAM_RANGE,
    max_features=CHAR_MAX_FEATURES,
    min_df=2,
    sublinear_tf=True
)

X_train_word = word_vectorizer.fit_transform(train_df[TEXT_COL])
X_valid_word = word_vectorizer.transform(valid_df[TEXT_COL])
X_test_word  = word_vectorizer.transform(test_df[TEXT_COL])

X_train_char = char_vectorizer.fit_transform(train_df[TEXT_COL])
X_valid_char = char_vectorizer.transform(valid_df[TEXT_COL])
X_test_char  = char_vectorizer.transform(test_df[TEXT_COL])

def count_exclamation(text: str) -> int:
    return text.count("!") + text.count("！")

def count_question(text: str) -> int:
    return text.count("?") + text.count("؟")

def count_hashtags(text: str) -> int:
    return text.count("#")

def count_mentions(text: str) -> int:
    return text.count("@")

def count_urls(text: str) -> int:
    return len(re.findall(r"http[s]?://|www\.", text))

def build_extra_features(df: pd.DataFrame):
    texts = df[TEXT_COL].tolist()

    emoji = np.array([emoji_flag(t) for t in texts]).reshape(-1, 1)
    char_len = np.array([len(t) for t in texts]).reshape(-1, 1)
    word_len = np.array([len(t.split()) for t in texts]).reshape(-1, 1)
    exclaim = np.array([count_exclamation(t) for t in texts]).reshape(-1, 1)
    question = np.array([count_question(t) for t in texts]).reshape(-1, 1)
    hashtags = np.array([count_hashtags(t) for t in texts]).reshape(-1, 1)
    mentions = np.array([count_mentions(t) for t in texts]).reshape(-1, 1)
    urls = np.array([count_urls(t) for t in texts]).reshape(-1, 1)

    extra = np.hstack([
        emoji,
        char_len,
        word_len,
        exclaim,
        question,
        hashtags,
        mentions,
        urls
    ])

    return csr_matrix(extra, dtype=np.float32)

X_train_extra = build_extra_features(train_df)
X_valid_extra = build_extra_features(valid_df)
X_test_extra  = build_extra_features(test_df)

X_train = hstack([X_train_word, X_train_char, X_train_extra]).tocsr()
X_valid = hstack([X_valid_word, X_valid_char, X_valid_extra]).tocsr()
X_test  = hstack([X_test_word, X_test_char, X_test_extra]).tocsr()

print("X_train shape:", X_train.shape)

# ============================================================
# 6) Build Model
# ============================================================
base_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LR,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE,
    reg_lambda=XGB_REG_LAMBDA,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model = MultiOutputClassifier(base_xgb)

# ============================================================
# 7) Train
# ============================================================
print("\n⏳ جاري التدريب (XGBoost Multilabel)...")
model.fit(X_train, y_train)

# ============================================================
# 8) Predict on Validation + Test
# ============================================================
y_valid_pred = model.predict(X_valid)
y_test_pred  = model.predict(X_test)

valid_probas = model.predict_proba(X_valid)
test_probas  = model.predict_proba(X_test)

y_valid_prob = np.column_stack([p[:, 1] for p in valid_probas])
y_test_prob  = np.column_stack([p[:, 1] for p in test_probas])

# ============================================================
# 9) Metrics
# ============================================================
def compute_metrics(y_true, y_pred):
    metrics = {}
    metrics["overall_acc"] = np.mean(np.all(y_true == y_pred, axis=1))

    metrics["dialect_acc"] = accuracy_score(y_true[:, 0], y_pred[:, 0])
    metrics["dialect_f1"] = f1_score(y_true[:, 0], y_pred[:, 0], average="macro", zero_division=0)
    metrics["dialect_f1_micro"] = f1_score(y_true[:, 0], y_pred[:, 0], average="micro", zero_division=0)

    metrics["hate_acc"] = accuracy_score(y_true[:, 1], y_pred[:, 1])
    metrics["hate_f1"] = f1_score(y_true[:, 1], y_pred[:, 1], average="macro", zero_division=0)
    metrics["hate_f1_micro"] = f1_score(y_true[:, 1], y_pred[:, 1], average="micro", zero_division=0)

    metrics["topic_acc"] = accuracy_score(y_true[:, 2], y_pred[:, 2])
    metrics["topic_f1"] = f1_score(y_true[:, 2], y_pred[:, 2], average="macro", zero_division=0)
    metrics["topic_f1_micro"] = f1_score(y_true[:, 2], y_pred[:, 2], average="micro", zero_division=0)

    metrics["avg_f1"] = (metrics["dialect_f1"] + metrics["hate_f1"] + metrics["topic_f1"]) / 3.0
    metrics["avg_f1_micro"] = (
        metrics["dialect_f1_micro"] + metrics["hate_f1_micro"] + metrics["topic_f1_micro"]
    ) / 3.0

    return metrics

valid_metrics = compute_metrics(y_valid, y_valid_pred)
test_metrics  = compute_metrics(y_test, y_test_pred)

print("\nValidation Metrics:")
for k, v in valid_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nTest Metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

print(f"\nOverall exact-match accuracy (all labels together): {np.mean(np.all(y_test == y_test_pred, axis=1)):.4f}")

# ============================================================
# 10) Detailed Reports
# ============================================================
def print_report_with_micro(y_true, y_pred, title: str):
    rep = classification_report(y_true, y_pred, digits=4, zero_division=0, output_dict=True)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    def row(k):
        return rep.get(k, {"precision": 0.0, "recall": 0.0, "f1-score": 0.0, "support": 0})

    r0 = row("0")
    r1 = row("1")
    rmacro = row("macro avg")
    rweight = row("weighted avg")
    total_support = int(r0["support"]) + int(r1["support"])

    print(f"\n--- {title} ---")
    print(f"{'':>14}{'precision':>10}{'recall':>10}{'f1-score':>10}{'f1-micro':>10}{'support':>10}")
    print(f"{'0.0':>14}{r0['precision']:>10.4f}{r0['recall']:>10.4f}{r0['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r0['support']):>10}")
    print(f"{'1.0':>14}{r1['precision']:>10.4f}{r1['recall']:>10.4f}{r1['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r1['support']):>10}")
    print()
    print(f"{'accuracy':>14}{'':>10}{'':>10}{acc:>10.4f}{f1_micro:>10.4f}{total_support:>10}")
    print(f"{'macro avg':>14}{rmacro['precision']:>10.4f}{rmacro['recall']:>10.4f}{rmacro['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rmacro['support']):>10}")
    print(f"{'weighted avg':>14}{rweight['precision']:>10.4f}{rweight['recall']:>10.4f}{rweight['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rweight['support']):>10}")

print_report_with_micro(y_test[:, 0], y_test_pred[:, 0], "Dialect report (1=saudi,0=egyptian)")
print_report_with_micro(y_test[:, 1], y_test_pred[:, 1], "Hate report (1=hate,0=not hate)")
print_report_with_micro(y_test[:, 2], y_test_pred[:, 2], "Topic report (1=religious,0=political)")

# ============================================================
# 11) Save Artifacts
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(word_vectorizer, os.path.join(OUTPUT_DIR, "word_vectorizer.joblib"))
joblib.dump(char_vectorizer, os.path.join(OUTPUT_DIR, "char_vectorizer.joblib"))
joblib.dump(model, os.path.join(OUTPUT_DIR, "xgb_multioutput_model.joblib"))
print("\nSaved to:", OUTPUT_DIR)

# ============================================================
# 12) Inference Helpers
# ============================================================
def build_single_extra_features(text: str):
    text = simple_clean_text(text)

    arr = np.array([[
        emoji_flag(text),
        len(text),
        len(text.split()),
        count_exclamation(text),
        count_question(text),
        count_hashtags(text),
        count_mentions(text),
        count_urls(text),
    ]], dtype=np.float32)

    return csr_matrix(arr)

def featurize_single_text(text: str):
    text = simple_clean_text(text)
    x_word = word_vectorizer.transform([text])
    x_char = char_vectorizer.transform([text])
    x_extra = build_single_extra_features(text)
    x = hstack([x_word, x_char, x_extra]).tocsr()
    return x, text

# ============================================================
# 13) Predict Function
# ============================================================
def predict(text: str, threshold: float = 0.5):
    x, clean_text = featurize_single_text(text)

    preds = model.predict(x)[0]
    probs = [clf.predict_proba(x)[0] for clf in model.estimators_]

    dialect_probs = probs[0]
    hate_probs = probs[1]
    topic_probs = probs[2]

    dialect_id = int(preds[0])
    hate_id = int(preds[1])
    topic_id = int(preds[2])

    return {
        "clean_text": clean_text,
        "dialect_id": dialect_id,
        "hate_id": hate_id,
        "topic_id": topic_id,
        "dialect_label": id2dialect[dialect_id],
        "hate_label": id2hate[hate_id],
        "topic_label": id2topic[topic_id],
        "dialect_probs": {
            "Egyptian": float(dialect_probs[0]),
            "Saudi": float(dialect_probs[1]),
        },
        "hate_probs": {
            "Not Hate": float(hate_probs[0]),
            "Hate": float(hate_probs[1]),
        },
        "topic_probs": {
            "Political": float(topic_probs[0]),
            "Religious": float(topic_probs[1]),
        },
        "dialect_confidence": float(dialect_probs[dialect_id]),
        "hate_confidence": float(hate_probs[hate_id]),
        "topic_confidence": float(topic_probs[topic_id]),
        "emoji_flag": int(emoji_flag(clean_text)),
    }

# ============================================================
# 14) Interactive Mode
# ============================================================
print("\n=== Interactive Mode ===")
print("Type: exit / quit / stop to end\n")

while True:
    text = input("Tweet> ").strip()
    if text.lower() in {"exit", "quit", "stop"}:
        print("Done.")
        break
    if not text:
        continue

    result = predict(text, threshold=0.5)

    print(
        f"Dialect: {result['dialect_label']} "
        f"(confidence={result['dialect_confidence']:.3f}, "
        f"Egyptian={result['dialect_probs']['Egyptian']:.3f}, "
        f"Saudi={result['dialect_probs']['Saudi']:.3f})"
    )
    print(
        f"Hate: {result['hate_label']} "
        f"(confidence={result['hate_confidence']:.3f}, "
        f"Not Hate={result['hate_probs']['Not Hate']:.3f}, "
        f"Hate={result['hate_probs']['Hate']:.3f}, "
        f"emoji_flag={result['emoji_flag']})"
    )
    print(
        f"Topic: {result['topic_label']} "
        f"(confidence={result['topic_confidence']:.3f}, "
        f"Political={result['topic_probs']['Political']:.3f}, "
        f"Religious={result['topic_probs']['Religious']:.3f})"
    )

📂 جاري تجهيز البيانات...
Rows: 2773 595 594
Train label counts:
dialect: {1: 1417, 0: 1356}
hate: {1: 1443, 0: 1330}
topic: {0: 1407, 1: 1366}
X_train shape: (2773, 29538)

⏳ جاري التدريب (XGBoost Multilabel)...

Validation Metrics:
overall_acc: 0.6387
dialect_acc: 0.8723
dialect_f1: 0.8722
dialect_f1_micro: 0.8723
hate_acc: 0.7630
hate_f1: 0.7629
hate_f1_micro: 0.7630
topic_acc: 0.9529
topic_f1: 0.9529
topic_f1_micro: 0.9529
avg_f1: 0.8627
avg_f1_micro: 0.8627

Test Metrics:
overall_acc: 0.6347
dialect_acc: 0.8721
dialect_f1: 0.8719
dialect_f1_micro: 0.8721
hate_acc: 0.7374
hate_f1: 0.7372
hate_f1_micro: 0.7374
topic_acc: 0.9697
topic_f1: 0.9697
topic_f1_micro: 0.9697
avg_f1: 0.8596
avg_f1_micro: 0.8597

Overall exact-match accuracy (all labels together): 0.6347

--- Dialect report (1=saudi,0=egyptian) ---
               precision    recall  f1-score  f1-micro   support
           0.0    0.9088    0.8459    0.8762    0.8721       318
           1.0    0.8356    0.9022    0.8676    0.8